# Can BERT keep up with the news?

**Cinthia Terceros · Virginia Tech CS 4804**

This notebook fine-tunes BERT on 2014–2016 headlines, selects a checkpoint on an early-period validation split, and evaluates it on two later periods.

| Test period | Headlines | Accuracy | Macro F1 | Hyperpartisan recall |
| --- | ---: | ---: | ---: | ---: |
| 2017–2020 | 1,018 | 75.9% | 75.8% | 73.8% |
| 2021–2022 | 447 | 73.8% | 71.9% | 58.5% |

These are saved results from the original course run. The public copy clears execution outputs and removes Google Drive export cells. The training run used a T4 GPU, Transformers 5.22.1, Datasets 4.8.5, PyTorch 2.14.1+cu130, and scikit-learn 1.6.1. Run top to bottom in a GPU-enabled Colab runtime. The setup downloads the public dataset and pretrained model. New runs may differ from the saved results.

[Live case study](https://ferchitp.github.io/bert-headline-evaluation/) · [Source dataset](https://github.com/VIStA-H/Hyperpartisan-News-Titles)


## Set up the Colab environment

In [ ]:
%pip install -q transformers==5.22.1 datasets==4.8.5 scikit-learn==1.6.1 accelerate matplotlib seaborn

## Prepare the data

In [ ]:

import os
import subprocess
from pathlib import Path

repo_url = "https://github.com/VIStA-H/Hyperpartisan-News-Titles.git"
repo_dir = Path("/content/Hyperpartisan-News-Titles")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True
    )

print("Dataset repository downloaded!")

data_dir = repo_dir / "data"

for file in sorted(data_dir.rglob("*")):
    if file.is_file():
        print(file.relative_to(repo_dir))


In [ ]:

import pandas as pd

# Load both manually labeled datasets
train_original = pd.read_csv(
    data_dir / "training_set.csv"
)

test_original = pd.read_csv(
    data_dir / "testing_set.csv"
)

# Combine them for our chronological experiment
df = pd.concat(
    [train_original, test_original],
    ignore_index=True
)

# Keep the columns needed for classification
df = df[["title", "year", "media", "label"]].copy()

# Remove missing titles or labels
df = df.dropna(subset=["title", "year", "label"])

df["title"] = df["title"].astype(str)
df["year"] = df["year"].astype(int)
df["label"] = df["label"].astype(int)

print("Total headlines:", len(df))
print("\nLabels:")
print(df["label"].value_counts())

print("\nHeadlines by year:")
print(df["year"].value_counts().sort_index())


In [ ]:

# Training period
early_df = df[df["year"].between(2014, 2016)].copy()

# First testing period
middle_df = df[df["year"].between(2017, 2020)].copy()

# Second testing period
late_df = df[df["year"].between(2021, 2022)].copy()

print("2014–2016:", len(early_df))
print("2017–2020:", len(middle_df))
print("2021–2022:", len(late_df))

print("\nLabel distribution by period:")

for name, subset in [
    ("2014–2016", early_df),
    ("2017–2020", middle_df),
    ("2021–2022", late_df)
]:
    print(f"\n{name}")
    print(subset["label"].value_counts())


In [ ]:

# Normalize headlines for duplicate checking
df["normalized_title"] = (
    df["title"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print(
    "Duplicate headline titles:",
    df["normalized_title"].duplicated().sum()
)

print("\nDuplicates across political eras:")

early_titles = set(
    df.loc[df["year"].between(2014, 2016),
           "normalized_title"]
)

middle_titles = set(
    df.loc[df["year"].between(2017, 2020),
           "normalized_title"]
)

late_titles = set(
    df.loc[df["year"].between(2021, 2022),
           "normalized_title"]
)

print(
    "Early vs Middle:",
    len(early_titles & middle_titles)
)

print(
    "Early vs Late:",
    len(early_titles & late_titles)
)

print(
    "Middle vs Late:",
    len(middle_titles & late_titles)
)


In [ ]:

# Check why 2 records were removed
raw_df = pd.concat(
    [train_original, test_original],
    ignore_index=True
)

print("Missing values:")
print(raw_df[["title", "year", "label"]].isna().sum())

print("\nRows with missing values:")
display(
    raw_df[
        raw_df[["title", "year", "label"]]
        .isna().any(axis=1)
    ]
)

print("\nDuplicate headlines:")
display(
    df[df["normalized_title"].duplicated(keep=False)]
    .sort_values("normalized_title")
    [["year", "media", "title", "label"]]
)


In [ ]:

from sklearn.model_selection import train_test_split

# Work from the cleaned dataset
clean_df = df.copy()

# Find normalized titles with conflicting labels
conflicting_titles = (
    clean_df.groupby("normalized_title")["label"]
    .nunique()
)

conflicting_titles = conflicting_titles[
    conflicting_titles > 1
].index

print("Conflicting titles:", len(conflicting_titles))

# Remove records with conflicting labels
clean_df = clean_df[
    ~clean_df["normalized_title"].isin(conflicting_titles)
].copy()

# Recreate chronological splits
early_df = clean_df[
    clean_df["year"].between(2014, 2016)
].copy()

middle_df = clean_df[
    clean_df["year"].between(2017, 2020)
].copy()

late_df = clean_df[
    clean_df["year"].between(2021, 2022)
].copy()

# Split early-period data for training and validation
bert_train_df, bert_val_df = train_test_split(
    early_df,
    test_size=0.20,
    random_state=42,
    stratify=early_df["label"]
)

bert_train_df = bert_train_df.reset_index(drop=True)
bert_val_df = bert_val_df.reset_index(drop=True)

bert_test_middle = middle_df.reset_index(drop=True)
bert_test_late = late_df.reset_index(drop=True)

print("Training:", len(bert_train_df))
print("Validation:", len(bert_val_df))
print("Test 2017–2020:", len(bert_test_middle))
print("Test 2021–2022:", len(bert_test_late))


## Check the chronological split

In [ ]:

datasets_to_check = {
    "Training": bert_train_df,
    "Validation": bert_val_df,
    "Test 2017–2020": bert_test_middle,
    "Test 2021–2022": bert_test_late
}

for name, df in datasets_to_check.items():
    print(f"\n{name}")
    print("Total headlines:", len(df))
    print("Years:", sorted(df["year"].unique()))
    print("Label distribution:")
    print(df["label"].value_counts().sort_index())

assert bert_train_df["year"].between(2014, 2016).all()
assert bert_val_df["year"].between(2014, 2016).all()
assert bert_test_middle["year"].between(2017, 2020).all()
assert bert_test_late["year"].between(2021, 2022).all()

print("\nAll chronological split checks passed!")


In [ ]:

def get_titles(df):
    return set(
        df["title"].dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )

train_titles = get_titles(bert_train_df)
val_titles = get_titles(bert_val_df)
middle_titles = get_titles(bert_test_middle)
late_titles = get_titles(bert_test_late)

checks = {
    "Training vs Validation":
        len(train_titles & val_titles),

    "Training vs Middle Test":
        len(train_titles & middle_titles),

    "Training vs Late Test":
        len(train_titles & late_titles),

    "Validation vs Middle Test":
        len(val_titles & middle_titles),

    "Validation vs Late Test":
        len(val_titles & late_titles),

    "Middle Test vs Late Test":
        len(middle_titles & late_titles)
}

for name, count in checks.items():
    print(f"{name}: {count} overlapping titles")

if all(count == 0 for count in checks.values()):
    print("\nNo exact headline overlap detected!")
else:
    print("\nReview overlapping headlines before submission.")


## Tokenize, fine-tune, and select a checkpoint

In [ ]:

from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

example = bert_train_df["title"].iloc[0]

tokens = tokenizer(
    example,
    truncation=True,
    padding="max_length",
    max_length=64
)

print("Example headline:", example)
print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(
    tokens["input_ids"]
))


In [ ]:

from datasets import Dataset

def prepare_dataset(dataframe):
    return Dataset.from_pandas(
        dataframe[["title", "label"]],
        preserve_index=False
    )

train_dataset = prepare_dataset(bert_train_df)
val_dataset = prepare_dataset(bert_val_df)
middle_dataset = prepare_dataset(bert_test_middle)
late_dataset = prepare_dataset(bert_test_late)

def tokenize_function(examples):
    return tokenizer(
        examples["title"],
        truncation=True,
        max_length=64
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

middle_dataset = middle_dataset.map(
    tokenize_function,
    batched=True
)

late_dataset = late_dataset.map(
    tokenize_function,
    batched=True
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Middle test dataset:", len(middle_dataset))
print("Late test dataset:", len(late_dataset))


In [ ]:

import torch
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "non-hyperpartisan",
        1: "hyperpartisan"
    },
    label2id={
        "non-hyperpartisan": 0,
        "hyperpartisan": 1
    }
)

print("BERT model loaded successfully!")
print("GPU available:", torch.cuda.is_available())
print("Number of labels:", model.config.num_labels)


In [ ]:

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels, predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

print("Evaluation metrics ready!")


In [ ]:

from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

training_args = TrainingArguments(
    output_dir="/content/bert_results",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=42,
    fp16=torch.cuda.is_available()
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting BERT training!")

trainer.train()

print("BERT training complete!")


## Evaluate and export predictions

In [ ]:

# Evaluate BERT on both political eras

middle_results = trainer.evaluate(
    eval_dataset=middle_dataset,
    metric_key_prefix="middle"
)

late_results = trainer.evaluate(
    eval_dataset=late_dataset,
    metric_key_prefix="late"
)

print("2017–2020 RESULTS")
for key, value in middle_results.items():
    print(f"{key}: {value}")

print("\n2021–2022 RESULTS")
for key, value in late_results.items():
    print(f"{key}: {value}")


In [ ]:

import pandas as pd

results_df = pd.DataFrame({
    "Period": ["2017–2020", "2021–2022"],
    "Accuracy": [
        middle_results["middle_accuracy"],
        late_results["late_accuracy"]
    ],
    "Macro Precision": [
        middle_results["middle_macro_precision"],
        late_results["late_macro_precision"]
    ],
    "Macro Recall": [
        middle_results["middle_macro_recall"],
        late_results["late_macro_recall"]
    ],
    "Macro F1": [
        middle_results["middle_macro_f1"],
        late_results["late_macro_f1"]
    ]
})

display(results_df.round(4))

results_df.to_csv(
    "/content/bert_results.csv",
    index=False
)

print("Results saved!")


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

bars = plt.bar(
    results_df["Period"],
    results_df["Macro F1"],
    color=["#4C78A8", "#F58518"]
)

plt.ylim(0, 1)
plt.ylabel("Macro-F1 Score")
plt.xlabel("Testing Period")
plt.title("BERT Performance Across Political Eras")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.02,
        f"{height:.3f}",
        ha="center"
    )

plt.tight_layout()

plt.savefig(
    "/content/bert_macro_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

periods = [
    ("2017–2020", middle_dataset, "middle"),
    ("2021–2022", late_dataset, "late")
]

prediction_tables = {}

for period, dataset, filename in periods:
    output = trainer.predict(dataset)

    predictions = np.argmax(
        output.predictions, axis=1
    )
    actual = np.array(dataset["label"])

    cm = confusion_matrix(
        actual, predictions, labels=[0, 1]
    )

    display_cm = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Non-hyperpartisan", "Hyperpartisan"]
    )

    fig, ax = plt.subplots(figsize=(7, 5))
    display_cm.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"BERT Confusion Matrix: {period}")
    plt.xticks(rotation=15)
    plt.tight_layout()

    plt.savefig(
        f"/content/bert_confusion_{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    print(f"\nClassification Report: {period}")
    print(classification_report(
        actual, predictions,
        target_names=["Non-hyperpartisan", "Hyperpartisan"],
        zero_division=0
    ))

    prediction_tables[filename] = pd.DataFrame({
        "title": dataset["title"],
        "actual": actual,
        "predicted": predictions
    })

    prediction_tables[filename].to_csv(
        f"/content/bert_predictions_{filename}.csv",
        index=False
    )
